# Climate Comfort Index · Random Forest Regressor

**36120 Advanced Machine Learning Application** · Assessment Task 2 · Weather Intelligence (Open-Meteo, Sydney)

*Experiment notebook · predicts CCI +1, +2 and +3 days ahead*

This notebook trains and evaluates the **Random Forest Regressor** model for the **Climate Comfort Index (CCI)**, using the leakage-audited feature matrix produced by the preprocessing notebook.

### Student details

| | |
|:--|:--|
| **Student name** | Juan Vargas T. |
| **Student ID** | 25945320 |
| **Subject** | 36120 Advanced Machine Learning Application |
| **Assessment** | AT2 · Weather Intelligence |
| **GitHub** | [Sebas102507](https://github.com/Sebas102507) |

## Overview

### Hypothesis

A Random Forest will **beat persistence and climatology at all three horizons** and prove more
robust than boosting at *t*+3, because averaging many decorrelated trees reduces variance without
chasing residual noise.

The competing possibility is that averaging washes out the weak frontal signal the delta and
pressure features were built to supply, leaving a smoothed climatology. The ablation (section 5)
and the prediction spread (section 8) test this directly.

### Design

| Choice | Value | Reason |
|:--|:--|:--|
| Algorithm | `RandomForestRegressor` | Low-variance ensemble with strong defaults on tabular data |
| Scaling | None | Trees split on thresholds, so scale is irrelevant |
| Strategy | One model per horizon (the endpoint returns three values) | Matches the deployed artefact layout |
| Selection | Optuna TPE sampler, 25 trials per target | See section 6 |
| Objective | validation **mean absolute error (MAE)**, in CCI points (lower is better) | See section 6 |

### Workflow

All ten experiment notebooks follow the same steps and run the same code from
`nucleus_ml.weather.Experiment`, so their results are directly comparable. **Only the model
specification in section 1 differs between notebooks.**

| # | Step | Purpose |
|:-:|:--|:--|
| 1 | Model specification | The only algorithm-specific cell |
| 2 | Data | Training and validation splits (the test split is never loaded here) |
| 3 | Baselines | The floor every model must clear |
| 4 | Default configuration | An untuned reference point |
| 5 | Feature-family ablation | What each block of engineered features contributes |
| 6 | Hyperparameter selection | Choose settings on the validation split |
| 7 | Seed stability | Is the tuning gain real or noise? |
| 8 | Final model | Tuned model against the default and the baselines |
| 9 | Feature importance | Which features the model relies on |
| 10 | Generalisation check | Training vs validation performance |
| 11 | Save artefacts | Model bundle + `/model-metadata` document |
| 12 | Conclusions | What to carry into the report |

---
## 0. Setup

Imports, the experiment's identity and the task definition. Everything else comes from the
project package, so no helper is redefined in this notebook.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

from nucleus_ml import weather as wx

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", "{:.3f}".format)

TASK = "comfort_climate"
EXPERIMENT = "cci_random_forest"
ALGORITHM = "Random Forest Regressor"
N_TRIALS = 25  # Optuna trials per target (None when a fixed grid is used)

task = wx.load_task(TASK)
print(f"{task.target_name} | {ALGORITHM} | {len(task.feature_columns)} features | "
      f"targets: {task.target_columns}")

---
## 1. Model specification

This is the **only algorithm-specific cell**. It defines how the estimator is built, a sensible
default configuration, a cheaper configuration for the ablation, and the candidate settings for
section 6 (either an Optuna search space or a fixed grid).

`max_features` matters most. The feature matrix contains blocks of near-duplicate columns (a
variable, its lags and its rolling means); low values force the trees to differ, which is what
makes bagging work. `min_samples_leaf` is the main complexity control on a dataset this small.

In [ ]:
def build_estimator(params: dict, seed: int) -> RandomForestRegressor:
    """A random forest regressor."""
    return RandomForestRegressor(n_jobs=-1, random_state=seed, **params)


DEFAULT_PARAMS = {"n_estimators": 400, "max_depth": None, "min_samples_leaf": 2,
                  "min_samples_split": 2, "max_features": 0.3}
ABLATION_PARAMS = {**DEFAULT_PARAMS, "n_estimators": 150}


def search_space(trial) -> dict:
    """Forest size, tree complexity and feature subsampling."""
    return {
        "n_estimators": trial.suggest_int("n_estimators", 150, 600, step=50),
        "max_depth": trial.suggest_categorical("max_depth", [None, 6, 10, 16, 24]),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 40),
        "max_features": trial.suggest_float("max_features", 0.05, 1.0),
    }


SEARCH_SPACE, GRID = search_space, None

In [ ]:
exp = wx.Experiment(
    task,
    name=EXPERIMENT,
    algorithm=ALGORITHM,
    build=build_estimator,
    default_params=DEFAULT_PARAMS,
    ablation_params=ABLATION_PARAMS,
    search_space=SEARCH_SPACE,
    grid=GRID,
)

---
## 2. Data

Only the **training** and **validation** splits are loaded. The test split is reserved for a
single evaluation in the comparison notebook, once one model has been selected, so nothing in
this notebook can be influenced by it.

> The preprocessing notebook already applied the chronological split, the warm-up filter and an
> embargo at each boundary. Nothing is re-split or shuffled here.

In [ ]:
exp.data_summary()

---
## 3. Baselines

Two naive forecasts, both fitted on the training split only:

| Baseline | Rule | What it knows |
|:--|:--|:--|
| **Persistence** | Tomorrow's CCI equals today's CCI | The current weather, but not the season |
| **Climatology** | The average CCI for that day of the year | The season, but not the current weather |

A model must beat **both** to be worth deploying: that shows it has combined current conditions
with the seasonal cycle.

In [ ]:
exp.run_baselines()

---
## 4. Default configuration

The untuned default is scored before any search. It is the reference for judging what tuning
bought in section 8; a search that fails to beat a sensible default is a result worth reporting.
`best_baseline` is repeated for comparison.

In [ ]:
exp.run_defaults()

---
## 5. Feature-family ablation

Feature families are added **one at a time**, starting from persistence alone (a single column),
and the model is refitted at each step. This measures what each block of engineering contributes
instead of assuming it. The cheaper `ABLATION_PARAMS` are used throughout so the differences come
from the features, not from tuning. Scores are on validation (MAE).

In [ ]:
ablation = exp.run_ablation()
plt.show()
ablation

---
## 6. Hyperparameter selection

The objective is validation **mean absolute error (MAE)**, in CCI points (lower is better).

**Why MAE.** It is measured in CCI points, so an improvement of 0.4 reads directly as "0.4 comfort
points closer on average". RMSE squares errors and would be dominated by the rare heavy-rain days
where CCI collapses, tuning the model for the tail instead of the ordinary days that make up
almost all API traffic.

**Why one search per horizon.** Predictability decays with lead time, so a single averaged score
would let a strong *t*+1 model hide a weak *t*+3 one.

**Why validation, not cross-validation.** The split is chronological. Shuffled k-fold would put
future days into training and inflate every score, because consecutive days are strongly
correlated. The fixed 2024 validation year matches how the model will be used: trained on the
past, applied forward.

> **Stated plainly:** selecting on validation makes the validation score slightly optimistic. The
> honest estimate comes from the test split, which only the comparison notebook reads, once.

In [ ]:
exp.tune(n_trials=N_TRIALS)

In [ ]:
exp.plot_search()
plt.show()

---
## 7. Seed stability

The selected configuration is refitted with three different random seeds. If the score moves by
as much as the spread between the top search candidates (`search_beats_noise = False`), the search
was ranking noise and the conclusion should say so. Deterministic models show a spread of zero.

In [ ]:
exp.seed_stability()

---
## 8. Final model against the baselines

The selected configuration is refitted and compared with the default configuration and every
baseline. `gain_of_tuned_model` is positive when the tuned model is better than that row.

`pred_std` is shown next to the error metrics. A spread far below the observed spread means the
model is playing safe by predicting close to the average, which can give a respectable MAE while
missing exactly the uncomfortable days users care about. Expect error to grow with the horizon.

In [ ]:
exp.finalise()

In [ ]:
exp.plot_validation()
plt.show()

---
## 9. Feature importance

Tree ensembles report **impurity-based importance** directly from the fitted model. Comparing horizons is informative: features that rise in rank from *t*+1 to *t*+3 carry genuine forecast signal rather than persistence.

> Correlated features (a variable, its lags and its rolling means) share importance, so a feature
> that looks unimportant on its own may belong to an essential block. The ablation in section 5 is
> the better guide to what can be removed.

In [ ]:
family_importance = exp.importance()
plt.show()
family_importance

---
## 10. Generalisation check

Training and validation scores side by side. A small gap means the model generalises; a large
gap means it has memorised patterns in the training years that do not carry over to 2024. Read
it together with the seed-stability table in section 7.

In [ ]:
exp.generalisation()

---
## 11. Save artefacts

The fitted model(s) are written to `models/comfort_climate/experiments/cci_random_forest/` with:

| File | Contents |
|:--|:--|
| `cci_random_forest_h<horizon>.joblib` | One fitted model per horizon |
| `metadata.json` | Target, algorithm, horizon, features, importance, hyperparameters, metrics — the `/model-metadata` document |
| `manifest.json` | File list and the feature column order the API must use |

> The feature column **order** is recorded because models receive an unnamed array at prediction
> time and would silently give wrong answers if the columns were permuted.

In [ ]:
exp.export()

---
## 12. Conclusions

- **Against the baselines.** Beating persistence at *t*+1 is the hardest test, because
  persistence is strongest at short lead times.
- **Against boosting.** If the forest wins at *t*+3 but loses at *t*+1, boosting exploits
  structure where it exists and fits noise where it does not — a signal-to-noise story worth
  telling in the report.
- **Spread.** Averaging hundreds of trees makes `pred_std` fall below the observed spread; report
  how far, because it limits how well the endpoint can flag genuinely uncomfortable days.

> The test split has still not been used. It is scored exactly once, in the
> `cci_comparison` notebook, after one model has been selected on validation
> performance across all five CCI experiments.